<a href="https://colab.research.google.com/github/ubaid8878/Flyrank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ubaid8878/Flyrank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
### Finding 1 — Growing pages are younger than declining pages

The paper reports that growing pages were younger on average than declining pages: about 185 days versus 228 days, while average word count was almost the same. The finding is useful as a descriptive comparison because it identifies age as a directional difference between the two groups.

Methodology question:
How exactly is the growing/declining label defined, and are the pages from the same clients represented across both groups? Since the label is based on traffic direction, I would want to confirm that the comparison is not being driven by differences in client mix or observation windows. The result supports an observed association between age and trend in this dataset, but it does not by itself establish that page age causes decline.

### Finding 2 — Refreshed mature pages show much higher impressions than stale pages

The paper reports a very large impression difference between refreshed mature pages and stale pages and presents freshness as a strong operational lever.

Methodology question:
How comparable were the refreshed and stale pages before the refresh? Pages selected for refreshing may already differ in age, visibility, topic, client, or prior performance. I would therefore ask whether the comparison controls for these differences and whether the validation design separates the effect of refreshing from selection effects. Statistical significance can show that groups differ, but it does not by itself prove that refreshing caused the difference.

Overall, these are useful findings to investigate, but I would treat them as observed and directional evidence rather than causal proof.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
## Honest validation design

My Week-5 model used a random row-level 80/20 split. That is useful as a first experiment, but pages from the same client could appear in both training and testing.

For this audit I use a client-grouped split. All pages belonging to a client stay together, so the model is tested on clients it did not see during training. This is a stricter test of whether the observed model advantage generalizes beyond the clients used for training.

I compare the old row-random result with the grouped result using Precision@20 and Precision@50. The Week-4 baseline is evaluated on the exact same grouped test rows.

In [3]:
import os
import sys
import subprocess

# Repository location
REPO_DIR = "/content/flyrank-ml-internship-starter"
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"

# Clone the starter repo if it is not already there
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

# Move into the repository
os.chdir(REPO_DIR)

print("Current folder:", os.getcwd())

# Check the dataset
DATA_PATH = "data/raw/content_refresh_anonymized.csv"

print("Dataset exists:", os.path.exists(DATA_PATH))

if not os.path.exists(DATA_PATH):
    print("\nFiles/folders here:")
    print(os.listdir())

    raise FileNotFoundError(
        "Dataset not found. Check the output above."
    )

print("Dataset found:", DATA_PATH)

Current folder: /content/flyrank-ml-internship-starter
Dataset exists: True
Dataset found: data/raw/content_refresh_anonymized.csv


In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

# Load the starter data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Target
y = df["trend_direction"].str.lower().eq("down").astype(int)

# Same observable features used in Week 5
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

# Client-grouped 80/20 split
groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_g = X.iloc[train_idx]
X_test_g = X.iloc[test_idx]

y_train_g = y.iloc[train_idx]
y_test_g = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training rows:", len(X_train_g))
print("Testing rows:", len(X_test_g))
print("Training clients:", groups_train.nunique())
print("Testing clients:", groups_test.nunique())
print("Training declining rate:", round(y_train_g.mean(), 3))
print("Testing declining rate:", round(y_test_g.mean(), 3))

print(
    "Client overlap:",
    len(set(groups_train) & set(groups_test))
)

Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Training declining rate: 0.55
Testing declining rate: 0.511
Client overlap: 0


In [5]:
# Train the same Week-5 Decision Tree
grouped_model = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

grouped_model.fit(X_train_g, y_train_g)

grouped_scores = grouped_model.predict_proba(X_test_g)[:, 1]


def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()


model_p20_grouped = precision_at_k(
    grouped_scores,
    y_test_g,
    20
)

model_p50_grouped = precision_at_k(
    grouped_scores,
    y_test_g,
    50
)

print(
    "Grouped Decision Tree Precision@20:",
    round(model_p20_grouped, 3)
)

print(
    "Grouped Decision Tree Precision@50:",
    round(model_p50_grouped, 3)
)

Grouped Decision Tree Precision@20: 0.55
Grouped Decision Tree Precision@50: 0.58


In [7]:
# Week-4 baseline on the exact same grouped test rows

test_df = df.iloc[test_idx].copy()

baseline_grouped_scores = (
    (test_df["days_since_last_update"] >= 180).astype(int) * 2
    + (test_df["ctr"] < df["ctr"].median()).astype(int)
)

baseline_p20_grouped = precision_at_k(
    baseline_grouped_scores,
    y_test_g,
    20
)

baseline_p50_grouped = precision_at_k(
    baseline_grouped_scores,
    y_test_g,
    50
)

print(
    "Grouped Baseline Precision@20:",
    round(baseline_p20_grouped, 3)
)

print(
    "Grouped Baseline Precision@50:",
    round(baseline_p50_grouped, 3)
)

Grouped Baseline Precision@20: 0.5
Grouped Baseline Precision@50: 0.66


In [8]:
comparison = pd.DataFrame({
    "Evaluation": [
        "Week-5 random split",
        "Week-5 random split",
        "Week-6 grouped split",
        "Week-6 grouped split"
    ],
    "Method": [
        "Decision Tree",
        "Week-4 Baseline",
        "Decision Tree",
        "Week-4 Baseline"
    ],
    "Precision@20": [
        0.65,
        0.50,
        model_p20_grouped,
        baseline_p20_grouped
    ],
    "Precision@50": [
        0.68,
        0.44,
        model_p50_grouped,
        baseline_p50_grouped
    ]
})

print(comparison.round(3).to_string(index=False))

          Evaluation          Method  Precision@20  Precision@50
 Week-5 random split   Decision Tree          0.65          0.68
 Week-5 random split Week-4 Baseline          0.50          0.44
Week-6 grouped split   Decision Tree          0.55          0.58
Week-6 grouped split Week-4 Baseline          0.50          0.66


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
## Leakage audit

The final feature set contains six observable signals: content_age_days, days_since_last_update, impressions_90d, avg_position, ctr, and word_count.

I excluded trend_direction and trend_pct because the target is derived from trend_direction and trend_pct is part of the same outcome calculation. I also excluded client_id and content_id because identifiers are for grouping or joins, not predictive features.

The validation split also prevents client overlap between training and testing. This reduces the risk that client-specific patterns are learned from one set of pages and evaluated on another set from the same client.

In [9]:
# Leakage audit

forbidden = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id",
    "content_id"
]

print("Final model features:")
print(features)

print("\nForbidden columns checked:")
for col in forbidden:
    print(f"{col}: {'USED' if col in features else 'NOT USED'}")

print("\nClient overlap between train and test:")
print(len(set(groups_train) & set(groups_test)))

assert "trend_direction" not in features
assert "trend_pct" not in features
assert "client_id" not in features
assert "content_id" not in features

assert len(set(groups_train) & set(groups_test)) == 0

print("\nLeakage audit: PASSED")

Final model features:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']

Forbidden columns checked:
trend_direction: NOT USED
trend_pct: NOT USED
is_declining_label: NOT USED
client_id: NOT USED
content_id: NOT USED

Client overlap between train and test:
0

Leakage audit: PASSED


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
## Claim rewrite

### Original Week-5 claim

The Decision Tree performed better than my Week-4 baseline.

### Safer claim

On my Week-5 random row-level test split, the Decision Tree measured higher Precision@20 and Precision@50 than the Week-4 baseline. After changing the validation design to a client-grouped split, I treat the result as a stricter test of generalization. The grouped result is directional evidence about how the model performs on unseen clients, not proof that the Decision Tree will always outperform the baseline.

### Decision-support interpretation

The model appears useful for prioritization on this dataset, but its value should be judged using grouped or time-aware validation and repeated testing rather than one random split. The results are measured evidence for this experiment, not a claim about Google's ranking algorithm.


In [10]:
# Real failure examples on the grouped test set

failure_df = test_df[
    [
        "content_id",
        "client_id",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "ctr",
        "word_count",
        "trend_direction"
    ]
].copy()

failure_df["actual"] = y_test_g.values
failure_df["model_score"] = grouped_scores
failure_df["predicted"] = (grouped_scores >= 0.5).astype(int)

failures = failure_df[
    failure_df["actual"] != failure_df["predicted"]
].copy()

print("Total grouped-test classification errors:", len(failures))

print("\nExample failures:")
print(
    failures.head(10).to_string(index=False)
)

Total grouped-test classification errors: 2682

Example failures:
          content_id         client_id  content_age_days  days_since_last_update  impressions_90d  avg_position  ctr  word_count trend_direction  actual  model_score  predicted
content_a1fb4e703a9e client_4e07408562               445                      25            15320          20.3 0.05      2481.0            down       1     0.473757          0
content_a5a2fbc76336 client_8527a891e2               238                     103              307          39.8 0.00      1342.0          stable       0     0.665169          1
content_2da6ae9d0882 client_e629fa6598               502                      20              297          13.9 0.34         NaN            down       1     0.473757          0
content_72c5c2d73e5a client_4e07408562               300                      13             2426          30.0 0.12      2686.0          stable       0     0.665169          1
content_bce275871a25 client_f369cb89fc           

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.